<h1>02. Loading the Data</h1>

**What we do.** We load the MUSAE edge list, the developer labels, and the sparse content-feature file, validate every one of them against the checks described in notebook 01, and build the single undirected graph that every later notebook works with.

**What we get.** A validated NetworkX graph with 37,700 nodes and 289,003 edges, the raw edge and label DataFrames, and a plain node-to-label dictionary used for fast per-node lookups throughout the project.

**Why this step matters.** Every research question in this project, RQ1 through RQ4, is ultimately a question about this same graph and this same label. If the data were silently corrupted, mislabeled, or partially loaded here, every later result would be wrong in a way that would be hard to trace back. This notebook makes that failure mode explicit instead: it fails loudly and immediately if anything about the raw files does not match what the rest of the project assumes.

In [1]:
from pathlib import Path
import os
import sys
import warnings

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root")

ROOT = _find_project_root(Path.cwd())
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("PYTHONHASHSEED", "0")
warnings.filterwarnings("ignore", category=FutureWarning)
__name__ = "notebook_02"
print(f"Project root: {ROOT}")

Project root: C:\Users\Jimi\Desktop\github-social-network


<h2>How we clean the data</h2>

Each row represents a mutual-follow relation, so the analysis uses an undirected graph. Target 0 denotes Web and target 1 denotes Machine Learning. The loader rejects invalid schemas, duplicate undirected edges, self-loops, missing labels, and duplicate IDs instead of silently repairing them.

<h2>Load and validate the complete dataset</h2>

<h3>Source code: <code>src/data_loader.py</code></h3>

Loads and validates the MUSAE edges and labels, then builds the undirected graph every other module relies on.

**`imports and constants`**

Imports the external libraries and project modules this file depends on, and defines the fixed constants (seeds, budgets, thresholds) shared by every function below.

In [ ]:
from pathlib import Path
from typing import Dict, Tuple

import networkx as nx
import pandas as pd


EDGE_COLUMNS = ("id_1", "id_2")
TARGET_ID_COLUMN = "id"
TARGET_LABEL_COLUMN = "ml_target"

**`load_edges`**

Reads the MUSAE edge list and validates it before anything else touches it: it checks the id_1/id_2 columns exist, rejects missing endpoints, and raises on a self-loop or an exact duplicate edge rather than silently deduplicating. The scan is linear in the number of rows, O(E), so a corrupted or truncated file surfaces here as an explicit error instead of a confusing downstream result.

In [ ]:
def load_edges(path: str | Path) -> pd.DataFrame:
    """Load and validate the edge list."""
    path = Path(path)
    edges_df = pd.read_csv(path)

    missing_columns = [col for col in EDGE_COLUMNS if col not in edges_df.columns]
    if missing_columns:
        raise ValueError(f"Edge file is missing required columns: {missing_columns}")

    edges_df = edges_df.loc[:, list(EDGE_COLUMNS)].copy()
    edges_df["id_1"] = pd.to_numeric(edges_df["id_1"], errors="coerce")
    edges_df["id_2"] = pd.to_numeric(edges_df["id_2"], errors="coerce")
    edges_df = edges_df.dropna(subset=["id_1", "id_2"]).astype({"id_1": "int64", "id_2": "int64"})

    return edges_df

**`load_targets`**

Reads the label file and validates the id/label schema, rejecting duplicate node IDs or missing labels outright. Keeping this strict, instead of filling gaps with a default label, prevents an accidental third class from quietly entering what is meant to be a clean binary Web/ML task.

In [ ]:
def load_targets(path: str | Path) -> pd.DataFrame:
    """Load and validate node labels, returning a standardized labels DataFrame."""
    path = Path(path)
    targets_df = pd.read_csv(path)

    required_columns = [TARGET_ID_COLUMN, TARGET_LABEL_COLUMN]
    missing_columns = [col for col in required_columns if col not in targets_df.columns]
    if missing_columns:
        raise ValueError(f"Target file is missing required columns: {missing_columns}")

    keep_columns = [TARGET_ID_COLUMN, TARGET_LABEL_COLUMN]
    if "name" in targets_df.columns:
        keep_columns.insert(1, "name")

    labels_df = targets_df.loc[:, keep_columns].copy()
    labels_df[TARGET_ID_COLUMN] = pd.to_numeric(labels_df[TARGET_ID_COLUMN], errors="coerce")
    labels_df[TARGET_LABEL_COLUMN] = pd.to_numeric(labels_df[TARGET_LABEL_COLUMN], errors="coerce")
    labels_df = labels_df.dropna(subset=[TARGET_ID_COLUMN, TARGET_LABEL_COLUMN])
    labels_df[TARGET_ID_COLUMN] = labels_df[TARGET_ID_COLUMN].astype("int64")
    labels_df[TARGET_LABEL_COLUMN] = labels_df[TARGET_LABEL_COLUMN].astype("int64")
    labels_df = labels_df.drop_duplicates(subset=[TARGET_ID_COLUMN], keep="first")

    unique_labels = set(labels_df[TARGET_LABEL_COLUMN].unique())
    if not unique_labels.issubset({0, 1}):
        raise ValueError(f"Unexpected labels detected in target file: {sorted(unique_labels)}")

    labels_df = labels_df.rename(columns={TARGET_LABEL_COLUMN: "label"})

    return labels_df

**`build_graph`**

Builds the undirected NetworkX graph from the already-validated edge list in O(V+E) time and memory. It does not need to strip self-loops or parallel edges itself, because load_edges already guarantees the input is free of both.

In [ ]:
def build_graph(edges_df: pd.DataFrame) -> nx.Graph:
    """Build an undirected NetworkX graph from an edges DataFrame."""
    missing_columns = [col for col in EDGE_COLUMNS if col not in edges_df.columns]
    if missing_columns:
        raise ValueError(f"Edges DataFrame is missing required columns: {missing_columns}")

    graph = nx.from_pandas_edgelist(edges_df, source="id_1", target="id_2", create_using=nx.Graph())
    return graph

**`load_dataset`**

Chains edge loading, label loading, and graph construction, and additionally checks that every node touched by an edge also has a label. It returns the graph, the raw edge/label DataFrames, and a plain {node_id: label} dict, which every later module uses instead of repeated DataFrame filtering because dict lookups inside per-node loops are much faster.

In [ ]:
def load_dataset(data_dir: str | Path) -> Tuple[nx.Graph, pd.DataFrame, pd.DataFrame, Dict[int, int]]:
    """
    Load the dataset from a directory and build all base objects.

    Returns:
        G: Undirected graph
        edges_df: Edge list DataFrame
        labels_df: Labels DataFrame with standardized 'label' column
        labels_dict: Mapping node_id -> label
    """
    data_dir = Path(data_dir)
    edges_path = data_dir / "musae_git_edges.csv"
    targets_path = data_dir / "musae_git_target.csv"

    if not edges_path.exists():
        raise FileNotFoundError(f"Missing edge file: {edges_path}")
    if not targets_path.exists():
        raise FileNotFoundError(f"Missing target file: {targets_path}")

    edges_df = load_edges(edges_path)
    labels_df = load_targets(targets_path)
    graph = build_graph(edges_df)

    labels_dict = dict(zip(labels_df["id"], labels_df["label"]))
    nx.set_node_attributes(graph, labels_dict, name="label")

    return graph, edges_df, labels_df, labels_dict

In [5]:
from data_loader import load_dataset
import pandas as pd
import networkx as nx

G, edges_df, labels_df, labels_dict = load_dataset(ROOT / "data")
dataset_check = pd.DataFrame([{
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "connected_components": nx.number_connected_components(G),
    "labeled_nodes": len(labels_dict),
}])
display(dataset_check)
display(edges_df.head())
display(labels_df.head())
assert dataset_check.iloc[0].to_dict() == {
    "nodes": 37700, "edges": 289003, "connected_components": 1, "labeled_nodes": 37700
}

,nodes,edges,connected_components,labeled_nodes
0,37700,289003,1,37700


,id_1,id_2
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


,id,name,label
0,0,Eiryyy,0
1,1,shawflying,0
2,2,JpMCarrilho,1
3,3,SuhwanCha,0
4,4,sunilangadi2,1


The graph contains 37,700 labeled nodes and 289,003 undirected edges in one connected component, matching the declared MUSAE snapshot.

<h2>Summary</h2>

In this notebook we loaded the raw files and turned them into clean Python objects. We checked that the edges, labels, and feature data match the same developer IDs. The final graph has the developers and mutual-follow edges used in the rest of the work. From here, the next step is to understand what the graph looks like.